# Кластеризация активов с 4 режимами (Bull/Bear × Low/High Vol)

ОСОБЕННОСТИ:
- 4 режима для Production (BULL_LOW, BULL_HIGH, BEAR_LOW, BEAR_HIGH)
- Архитектура готова к 9 режимам (словари, конфиги)
- Кластеры меняются квартально, параметры обновляются ежедневно
- 16 unit-тестов включая интеграционный

## 1. УСТАНОВКА ЗАВИСИМОСТЕЙ

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import logging
from datetime import datetime, timedelta
from scipy.linalg import cholesky
from scipy.stats import wasserstein_distance
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage
from joblib import Parallel, delayed, Memory
import warnings
warnings.filterwarnings('ignore')

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('clustering.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

## 2. КОНФИГУРАЦИЯ И ПАРАМЕТРЫ

In [3]:
CLUSTERING_CONFIG = {
    # Группы истории
    'history_groups': {
        'A': {'min_days': 0, 'max_days': 30, 'method': 'sector_proxy_market'},
        'B': {'min_days': 30, 'max_days': 100, 'method': 'wasserstein'},
        'C': {'min_days': 100, 'max_days': 200, 'method': 'pearson_sma'},
        'D': {'min_days': 200, 'max_days': None, 'method': 'pearson_ewma'}
    },

    # Конфигурация режимов (поддержка 9, активны 4)
    'regimes': {
        # Активные режимы для Production (4 режима)
        'active': ['BULL_LOW', 'BULL_HIGH', 'BEAR_LOW', 'BEAR_HIGH'],

        # Все возможные режимы (архитектура для 9 режимов)
        'all': [
            'BULL_LOW', 'BULL_NORMAL', 'BULL_HIGH',
            'NEUTRAL_LOW', 'NEUTRAL_NORMAL', 'NEUTRAL_HIGH',
            'BEAR_LOW', 'BEAR_NORMAL', 'BEAR_HIGH'
        ],

        # Параметры для каждого режима
        'parameters': {
            # Bull режимы
            'BULL_LOW': {
                'direction': 'bull',
                'volatility': 'low',
                'drift_multiplier': 1.20,
                'sigma_multiplier': 0.80,
                'correlation_adjustment': -0.10,
                'haircut_multiplier': 0.90,
                'liquidity_horizon_base': 10
            },
            'BULL_NORMAL': {
                'direction': 'bull',
                'volatility': 'normal',
                'drift_multiplier': 1.00,
                'sigma_multiplier': 1.00,
                'correlation_adjustment': 0.00,
                'haircut_multiplier': 1.00,
                'liquidity_horizon_base': 10
            },
            'BULL_HIGH': {
                'direction': 'bull',
                'volatility': 'high',
                'drift_multiplier': 0.80,
                'sigma_multiplier': 1.30,
                'correlation_adjustment': 0.15,
                'haircut_multiplier': 1.20,
                'liquidity_horizon_base': 15
            },

            # Neutral режимы (не активны в Production)
            'NEUTRAL_LOW': {
                'direction': 'neutral',
                'volatility': 'low',
                'drift_multiplier': 1.00,
                'sigma_multiplier': 0.90,
                'correlation_adjustment': 0.00,
                'haircut_multiplier': 1.00,
                'liquidity_horizon_base': 10
            },
            'NEUTRAL_NORMAL': {
                'direction': 'neutral',
                'volatility': 'normal',
                'drift_multiplier': 1.00,
                'sigma_multiplier': 1.00,
                'correlation_adjustment': 0.00,
                'haircut_multiplier': 1.00,
                'liquidity_horizon_base': 10
            },
            'NEUTRAL_HIGH': {
                'direction': 'neutral',
                'volatility': 'high',
                'drift_multiplier': 0.90,
                'sigma_multiplier': 1.20,
                'correlation_adjustment': 0.10,
                'haircut_multiplier': 1.10,
                'liquidity_horizon_base': 12
            },

            # Bear режимы
            'BEAR_LOW': {
                'direction': 'bear',
                'volatility': 'low',
                'drift_multiplier': 0.70,
                'sigma_multiplier': 1.00,
                'correlation_adjustment': 0.10,
                'haircut_multiplier': 1.10,
                'liquidity_horizon_base': 12
            },
            'BEAR_NORMAL': {
                'direction': 'bear',
                'volatility': 'normal',
                'drift_multiplier': 0.60,
                'sigma_multiplier': 1.20,
                'correlation_adjustment': 0.15,
                'haircut_multiplier': 1.25,
                'liquidity_horizon_base': 15
            },
            'BEAR_HIGH': {
                'direction': 'bear',
                'volatility': 'high',
                'drift_multiplier': 0.50,
                'sigma_multiplier': 1.50,
                'correlation_adjustment': 0.25,
                'haircut_multiplier': 1.50,
                'liquidity_horizon_base': 20
            }
        }
    },

    # Определение режимов (Multi-Indicator + Volatility Overlay)
    'regime_detection': {
        # Направления (Bull/Bear)
        'direction': {
            'indicators': {
                'price_vs_sma200': {'weight': 0.30, 'bull_threshold': 1.05, 'bear_threshold': 0.95},
                'sma50_vs_sma200': {'weight': 0.20, 'bull_threshold': 1.02, 'bear_threshold': 0.98},
                'market_breadth': {'weight': 0.20, 'bull_threshold': 0.60, 'bear_threshold': 0.40},
                'volume': {'weight': 0.15, 'bull_threshold': 1.20, 'bear_threshold': 0.80},
                'macro': {'weight': 0.15, 'bull_threshold': 0.60, 'bear_threshold': 0.40}
            },
            'bull_score_threshold': 0.60,
            'bear_score_threshold': 0.40,
            'hysteresis_days': 3  # Нужно N дней подряд для смены режима
        },

        # Волатильность (Low/High)
        'volatility': {
            'vix_thresholds': {'low': 20, 'high': 35},
            'hist_vol_thresholds': {'low': 0.15, 'high': 0.25},
            'vix_weight': 0.60,
            'hist_vol_weight': 0.40
        }
    },

    # Корреляции
    'correlation': {
        'ewma_lambda': 0.94,
        'ewma_window': 504,
        'sma_window': 50,
        'min_sector_size': 3,
        'liquid_illiquid_cap': 0.75,
        'min_corr_window': 30
    },

    # Кластеризация
    'clustering': {
        'k_min': 2,
        'k_max': 15,
        'k_target': 7,
        'ari_threshold': 0.7,
        'silhouette_threshold': 0.3,
        'recluster_frequency_days': 63  # Квартальная перекластеризация
    },

    # Консервативные параметры
    'conservative': {
        'group_A': {'beta_multiplier': 1.30, 'sigma_multiplier': 1.20},
        'group_B': {'beta_multiplier': 1.20, 'sigma_multiplier': 1.10},
        'group_C': {'beta_multiplier': 1.05, 'sigma_multiplier': 1.05},
        'group_D': {'beta_multiplier': 1.00, 'sigma_multiplier': 1.00}
    },

    # Ликвидность
    'liquidity': {
        'threshold': 0.80,
        'imputation_beta_clip': (0.5, 3.0),
        'correlation_cap': 0.75,
        'liquidity_premium': {0.80: 1.0, 0.60: 1.2, 0.40: 1.5, 0.0: 2.0}
    },

    # Кризис
    'crisis': {
        'vix_threshold': 40,
        'imoex_drawdown': -0.20,
        'correlation_spike': 0.90,
        'freeze_duration_days': 63
    },

    # Производительность
    'performance': {
        'max_execution_hours': 2,
        'max_memory_gb': 8,
        'n_jobs': -1,
        'wasserstein_cache_size': 10000
    },

    # Валидация
    'validation': {
        'min_history_days': 30,
        'max_missing_ratio': 0.50,
        'min_liquidity_ratio': 0.20
    }
}

logger.info("Конфигурация загружена")
logger.info(f"Активные режимы: {CLUSTERING_CONFIG['regimes']['active']}")
logger.info(f"Поддерживается режимов: {len(CLUSTERING_CONFIG['regimes']['all'])}")

## 3. КЛАССИФИКАЦИЯ ДАННЫХ

In [4]:
class DataClassifier:
    """Классификация инструментов по истории и ликвидности"""

    def __init__(self, config):
        self.config = config
        logger.info("DataClassifier инициализирован")

    def validate_data(self, returns_df, reference_date):
        """Валидация входных данных"""
        validation_results = {'valid': True, 'errors': [], 'warnings': []}

        if returns_df.empty:
            validation_results['valid'] = False
            validation_results['errors'].append("Пустой DataFrame")
            return validation_results

        if not isinstance(returns_df.index, pd.DatetimeIndex):
            validation_results['warnings'].append("Индекс не DatetimeIndex")

        missing_ratio = returns_df.isna().sum().sum() / (len(returns_df) * len(returns_df.columns))
        if missing_ratio > self.config['validation']['max_missing_ratio']:
            validation_results['warnings'].append(f"Высокая доля пропусков: {missing_ratio:.1%}")

        for ticker in returns_df.columns:
            available_data = returns_df[ticker].loc[:reference_date]
            n_days = available_data.notna().sum()

            if n_days < self.config['validation']['min_history_days']:
                validation_results['warnings'].append(f"{ticker}: недостаточно истории ({n_days} дней)")

            liq_ratio = n_days / max(1, len(available_data))
            if liq_ratio < self.config['validation']['min_liquidity_ratio']:
                validation_results['warnings'].append(f"{ticker}: низкая ликвидность ({liq_ratio:.1%})")

        return validation_results

    def classify_by_history(self, returns_df, reference_date):
        """Классификация по длине истории"""
        history_groups = {'A': [], 'B': [], 'C': [], 'D': []}

        for ticker in returns_df.columns:
            available_data = returns_df[ticker].loc[:reference_date]
            n_days = available_data.notna().sum()

            if n_days < self.config['history_groups']['A']['max_days']:
                history_groups['A'].append(ticker)
            elif n_days < self.config['history_groups']['B']['max_days']:
                history_groups['B'].append(ticker)
            elif n_days < self.config['history_groups']['C']['max_days']:
                history_groups['C'].append(ticker)
            else:
                history_groups['D'].append(ticker)

        logger.info(f"Классификация по истории: A={len(history_groups['A'])}, B={len(history_groups['B'])}, C={len(history_groups['C'])}, D={len(history_groups['D'])}")

        return history_groups

    def classify_by_liquidity(self, returns_df, reference_date):
        """Классификация по ликвидности"""
        threshold = self.config['liquidity']['threshold']
        liquid = []
        illiquid = []

        for ticker in returns_df.columns:
            available_data = returns_df[ticker].loc[:reference_date]
            n_trading_days = available_data.notna().sum()
            n_calendar_days = len(available_data)
            liquidity_ratio = n_trading_days / max(1, n_calendar_days)

            if liquidity_ratio >= threshold:
                liquid.append(ticker)
            else:
                illiquid.append(ticker)

        logger.info(f"Классификация по ликвидности: {len(liquid)} ликвидных, {len(illiquid)} неликвидных")

        return liquid, illiquid

    def get_liquidity_ratio(self, returns_df, ticker, reference_date):
        """Расчёт коэффициента ликвидности для тикера"""
        available_data = returns_df[ticker].loc[:reference_date]
        n_trading_days = available_data.notna().sum()
        n_calendar_days = len(available_data)
        return n_trading_days / max(1, n_calendar_days)

## 4. ОПРЕДЕЛЕНИЕ РЫНОЧНОГО РЕЖИМА (4 РЕЖИМА)

In [5]:
class RegimeDetector:
    """
    Определение режимов: Bull/Bear × Low/High Vol
    Поддерживает архитектуру для 9 режимов, активны 4
    """

    def __init__(self, imoex_data, config):
        self.imoex_data = imoex_data
        self.config = config
        self.regime_history = []  # Для гистерезиса
        logger.info("RegimeDetector инициализирован (4 режима)")

    def calculate_direction_score(self, date, market_returns=None):
        """
        Расчёт Score направления рынка (Multi-Indicator)
        Returns: score от 0.0 (медвежий) до 1.0 (бычий)
        """
        indicators_config = self.config['regime_detection']['direction']['indicators']
        scores = {}

        # Индикатор 1: Цена vs SMA-200
        imoex_window = self.imoex_data.loc[:date].tail(200)
        if len(imoex_window) >= 200:
            sma_200 = imoex_window['close'].mean()
            current_price = self.imoex_data.loc[date, 'close']
            ratio = current_price / sma_200

            if ratio >= indicators_config['price_vs_sma200']['bull_threshold']:
                scores['price_vs_sma200'] = 1.0
            elif ratio <= indicators_config['price_vs_sma200']['bear_threshold']:
                scores['price_vs_sma200'] = 0.0
            else:
                scores['price_vs_sma200'] = (ratio - indicators_config['price_vs_sma200']['bear_threshold']) / \
                                           (indicators_config['price_vs_sma200']['bull_threshold'] - indicators_config['price_vs_sma200']['bear_threshold'])
        else:
            scores['price_vs_sma200'] = 0.5

        # Индикатор 2: SMA-50 vs SMA-200
        if len(imoex_window) >= 200:
            sma_50 = imoex_window['close'].tail(50).mean()
            ratio = sma_50 / sma_200

            if ratio >= indicators_config['sma50_vs_sma200']['bull_threshold']:
                scores['sma50_vs_sma200'] = 1.0
            elif ratio <= indicators_config['sma50_vs_sma200']['bear_threshold']:
                scores['sma50_vs_sma200'] = 0.0
            else:
                scores['sma50_vs_sma200'] = (ratio - indicators_config['sma50_vs_sma200']['bear_threshold']) / \
                                           (indicators_config['sma50_vs_sma200']['bull_threshold'] - indicators_config['sma50_vs_sma200']['bear_threshold'])
        else:
            scores['sma50_vs_sma200'] = 0.5

        # Индикатор 3: Ширина рынка (если есть market_returns)
        if market_returns is not None:
            breadth = (market_returns > 0).sum() / max(1, len(market_returns))
            if breadth >= indicators_config['market_breadth']['bull_threshold']:
                scores['market_breadth'] = 1.0
            elif breadth <= indicators_config['market_breadth']['bear_threshold']:
                scores['market_breadth'] = 0.0
            else:
                scores['market_breadth'] = (breadth - indicators_config['market_breadth']['bear_threshold']) / \
                                          (indicators_config['market_breadth']['bull_threshold'] - indicators_config['market_breadth']['bear_threshold'])
        else:
            scores['market_breadth'] = 0.5

        # Индикатор 4: Объём (упрощённо - волатильность)
        if len(imoex_window) >= 50:
            recent_vol = imoex_window['close'].tail(20).pct_change().std()
            older_vol = imoex_window['close'].tail(50).head(30).pct_change().std()
            volume_ratio = recent_vol / max(older_vol, 1e-10)

            if volume_ratio >= indicators_config['volume']['bull_threshold']:
                scores['volume'] = 1.0
            elif volume_ratio <= indicators_config['volume']['bear_threshold']:
                scores['volume'] = 0.0
            else:
                scores['volume'] = (volume_ratio - indicators_config['volume']['bear_threshold']) / \
                                  (indicators_config['volume']['bull_threshold'] - indicators_config['volume']['bear_threshold'])
        else:
            scores['volume'] = 0.5

        # Индикатор 5: Макро (упрощённо - тренд)
        if len(imoex_window) >= 200:
            trend = (imoex_window['close'].iloc[-1] - imoex_window['close'].iloc[0]) / imoex_window['close'].iloc[0]
            if trend >= indicators_config['macro']['bull_threshold']:
                scores['macro'] = 1.0
            elif trend <= indicators_config['macro']['bear_threshold']:
                scores['macro'] = 0.0
            else:
                scores['macro'] = (trend - indicators_config['macro']['bear_threshold']) / \
                                 (indicators_config['macro']['bull_threshold'] - indicators_config['macro']['bear_threshold'])
        else:
            scores['macro'] = 0.5

        # Взвешенный Score
        total_score = 0.0
        total_weight = 0.0
        for indicator, score in scores.items():
            weight = indicators_config[indicator]['weight']
            total_score += score * weight
            total_weight += weight

        return total_score / max(total_weight, 1e-10), scores

    def calculate_volatility_score(self, date, vix_data=None):
        """
        Расчёт Score волатильности (VIX-RUS + Историческая волатильность)
        Returns: score от 0.0 (Low Vol) до 1.0 (High Vol)
        """
        vol_config = self.config['regime_detection']['volatility']

        # VIX-RUS (если доступен)
        if vix_data is not None and date in vix_data.index:
            vix = vix_data.loc[date, 'value']
            if vix <= vol_config['vix_thresholds']['low']:
                vix_score = 0.0
            elif vix >= vol_config['vix_thresholds']['high']:
                vix_score = 1.0
            else:
                vix_score = (vix - vol_config['vix_thresholds']['low']) / \
                           (vol_config['vix_thresholds']['high'] - vol_config['vix_thresholds']['low'])
        else:
            vix_score = 0.5  # Fallback

        # Историческая волатильность IMOEX
        imoex_window = self.imoex_data.loc[:date].tail(20)
        if len(imoex_window) >= 20:
            hist_vol = imoex_window['close'].pct_change().std() * np.sqrt(252)
            if hist_vol <= vol_config['hist_vol_thresholds']['low']:
                hist_score = 0.0
            elif hist_vol >= vol_config['hist_vol_thresholds']['high']:
                hist_score = 1.0
            else:
                hist_score = (hist_vol - vol_config['hist_vol_thresholds']['low']) / \
                            (vol_config['hist_vol_thresholds']['high'] - vol_config['hist_vol_thresholds']['low'])
        else:
            hist_score = 0.5

        # Взвешенный Score
        total_score = (vix_score * vol_config['vix_weight'] +
                      hist_score * vol_config['hist_vol_weight'])

        return total_score, {'vix_score': vix_score, 'hist_score': hist_score}

    def detect_regime(self, date, vix_data=None, market_returns=None):
        """
        Определение режима: Bull/Bear × Low/High Vol
        Returns: regime (строка), metrics (словарь)
        """
        # Расчёт Scores
        direction_score, direction_details = self.calculate_direction_score(date, market_returns)
        volatility_score, volatility_details = self.calculate_volatility_score(date, vix_data)

        # Определение направления с гистерезисом
        direction_config = self.config['regime_detection']['direction']

        if direction_score >= direction_config['bull_score_threshold']:
            new_direction = 'BULL'
        elif direction_score <= direction_config['bear_score_threshold']:
            new_direction = 'BEAR'
        else:
            new_direction = 'NEUTRAL'

        # Гистерезис (нужно N дней подряд для смены)
        self.regime_history.append(new_direction)
        if len(self.regime_history) > direction_config['hysteresis_days']:
            self.regime_history.pop(0)

        # Проверка устойчивости направления
        if len(self.regime_history) >= direction_config['hysteresis_days']:
            if all(r == 'BULL' for r in self.regime_history):
                direction = 'BULL'
            elif all(r == 'BEAR' for r in self.regime_history):
                direction = 'BEAR'
            else:
                direction = 'NEUTRAL'
        else:
            direction = new_direction

        # Определение волатильности
        vol_config = self.config['regime_detection']['volatility']
        if volatility_score <= 0.33:
            volatility = 'LOW'
        elif volatility_score >= 0.67:
            volatility = 'HIGH'
        else:
            volatility = 'NORMAL'

        # Формирование режима
        regime = f"{direction}_{volatility}"

        # ← Проверка: активен ли режим для Production
        active_regimes = self.config['regimes']['active']
        if regime not in active_regimes:
            # Fallback на ближайший активный режим
            if direction == 'NEUTRAL':
                direction = 'BULL' if direction_score > 0.5 else 'BEAR'
            if volatility == 'NORMAL':
                volatility = 'LOW' if volatility_score < 0.5 else 'HIGH'
            regime = f"{direction}_{volatility}"

        metrics = {
            'regime': regime,
            'direction': direction,
            'volatility': volatility,
            'direction_score': direction_score,
            'volatility_score': volatility_score,
            'direction_details': direction_details,
            'volatility_details': volatility_details,
            'is_active': regime in active_regimes
        }

        logger.info(f"Режим на {date}: {regime} (Dir={direction_score:.2f}, Vol={volatility_score:.2f})")

        return regime, metrics

    def is_crisis(self, metrics, avg_correlation=None):
        """Проверка на кризисный режим"""
        crisis_config = self.config['crisis']
        crisis_signals = []

        if metrics.get('volatility_details', {}).get('vix_score', 0) > 0.8:
            crisis_signals.append('vix')
            logger.warning(f"VIX выше порога")

        if metrics.get('imoex_drawdown') and metrics['imoex_drawdown'] < crisis_config['imoex_drawdown']:
            crisis_signals.append('drawdown')
            logger.warning(f"Просадка IMOEX: {metrics['imoex_drawdown']:.1%}")

        if avg_correlation and avg_correlation > crisis_config['correlation_spike']:
            crisis_signals.append('correlation')
            logger.warning(f"Корреляция выше порога: {avg_correlation:.2f}")

        is_crisis = len(crisis_signals) > 0

        if is_crisis:
            logger.warning(f"КРИЗИСНЫЙ РЕЖИМ обнаружен: {crisis_signals}")

        return is_crisis, crisis_signals

    def get_regime_parameters(self, regime):
        """Получение параметров для режима"""
        regime_params = self.config['regimes']['parameters'].get(regime, {})

        if not regime_params:
            # Fallback на ближайший активный режим
            active_regimes = self.config['regimes']['active']
            regime_params = self.config['regimes']['parameters'].get(active_regimes[0], {})
            logger.warning(f"Режим {regime} не найден, используем fallback: {active_regimes[0]}")

        return regime_params

## 5. РАСЧЁТ КОРРЕЛЯЦИЙ

In [6]:
class CorrelationCalculator:
    """Расчёт корреляций с разными методами для разных групп"""

    def __init__(self, config):
        self.config = config
        self.wasserstein_cache = {}
        logger.info("CorrelationCalculator инициализирован")

    def calculate_ewma_correlation(self, returns_df, ticker1, ticker2, reference_date):
        """EWMA корреляция для Группы D"""
        window = self.config['correlation']['ewma_window']
        lam = self.config['correlation']['ewma_lambda']
        min_window = self.config['correlation']['min_corr_window']

        data = returns_df.loc[:reference_date].tail(window)
        x = data[ticker1].dropna().values
        y = data[ticker2].dropna().values

        if len(x) < min_window or len(y) < min_window:
            return np.nan

        min_len = min(len(x), len(y))
        x = x[-min_len:]
        y = y[-min_len:]

        weights = np.array([lam ** i for i in range(min_len-1, -1, -1)])
        weights = weights / weights.sum()

        x_mean = np.sum(x * weights)
        y_mean = np.sum(y * weights)

        cov = np.sum(weights * (x - x_mean) * (y - y_mean))
        var_x = np.sum(weights * (x - x_mean) ** 2)
        var_y = np.sum(weights * (y - y_mean) ** 2)

        if var_x <= 0 or var_y <= 0:
            return np.nan

        corr = cov / np.sqrt(var_x * var_y)
        return np.clip(corr, -1.0, 1.0)

    def calculate_sma_correlation(self, returns_df, ticker1, ticker2, reference_date):
        """SMA корреляция для Группы C"""
        window = self.config['correlation']['sma_window']
        min_window = self.config['correlation']['min_corr_window']

        data = returns_df.loc[:reference_date].tail(window)
        x = data[ticker1].dropna().values
        y = data[ticker2].dropna().values

        if len(x) < min_window or len(y) < min_window:
            return np.nan

        min_len = min(len(x), len(y))
        x = x[-min_len:]
        y = y[-min_len:]

        try:
            corr = np.corrcoef(x, y)[0, 1]
            return np.clip(corr, -1.0, 1.0) if not np.isnan(corr) else np.nan
        except:
            return np.nan

    def calculate_wasserstein_distance(self, returns_df, ticker1, ticker2, reference_date):
        """Wasserstein расстояние для Группы B (с кэшированием)"""
        min_window = self.config['correlation']['min_corr_window']

        cache_key = (ticker1, ticker2, reference_date)
        if cache_key in self.wasserstein_cache:
            return self.wasserstein_cache[cache_key]

        data = returns_df.loc[:reference_date]
        x = data[ticker1].dropna().values
        y = data[ticker2].dropna().values

        if len(x) < min_window or len(y) < min_window:
            return np.nan

        x_norm = (x - np.mean(x)) / (np.std(x) + 1e-10)
        y_norm = (y - np.mean(y)) / (np.std(y) + 1e-10)

        try:
            distance = wasserstein_distance(x_norm, y_norm)
        except:
            distance = np.nan

        if len(self.wasserstein_cache) < self.config['performance']['wasserstein_cache_size']:
            self.wasserstein_cache[cache_key] = distance

        return distance

    def calculate_market_correlation(self, returns_df, ticker, market_returns, reference_date):
        """Расчёт корреляции с рынком для Группы A"""
        min_window = self.config['correlation']['min_corr_window']

        data = returns_df[ticker].loc[:reference_date]
        x = market_returns.reindex(data.index).dropna().values
        y = data.dropna().values

        if len(x) < min_window or len(y) < min_window:
            return np.nan

        min_len = min(len(x), len(y))
        x = x[-min_len:]
        y = y[-min_len:]

        try:
            corr = np.corrcoef(x, y)[0, 1]
            return np.clip(corr, -1.0, 1.0) if not np.isnan(corr) else np.nan
        except:
            return np.nan

    def get_sector_proxy_correlation(self, ticker1, ticker2, sector_map, market_corr=None):
        """Секторный proxy для Группы A с учётом корреляции с рынком"""
        sector1 = sector_map.get(ticker1, 'Unknown')
        sector2 = sector_map.get(ticker2, 'Unknown')

        if sector1 == sector2:
            base_corr = 0.70
        else:
            base_corr = 0.50

        if market_corr is not None and not np.isnan(market_corr):
            weight = min(1.0, max(0.3, market_corr))
            base_corr = base_corr * weight + 0.50 * (1 - weight)

        return base_corr

    def adjust_correlations_for_regime(self, corr_matrix, regime_params):
        """
        Корректировка корреляций в зависимости от режима
        """
        adjustment = regime_params.get('correlation_adjustment', 0.0)

        if adjustment == 0.0:
            return corr_matrix

        adjusted_matrix = corr_matrix.copy()
        n = adjusted_matrix.shape[0]

        for i in range(n):
            for j in range(i+1, n):
                adjusted_value = min(1.0, max(-1.0, corr_matrix[i, j] + adjustment))
                adjusted_matrix[i, j] = adjusted_value
                adjusted_matrix[j, i] = adjusted_value

        np.fill_diagonal(adjusted_matrix, 1.0)

        return adjusted_matrix

    def build_correlation_matrix(self, returns_df, tickers, reference_date,
                                  method, sector_map=None, market_returns=None):
        """Построение корреляционной матрицы"""
        n = len(tickers)
        corr_matrix = np.eye(n)

        if method == 'wasserstein':
            pairs = [(i, j) for i in range(n) for j in range(i+1, n)]

            def calc_pair(i, j):
                dist = self.calculate_wasserstein_distance(
                    returns_df, tickers[i], tickers[j], reference_date
                )
                if np.isnan(dist):
                    return i, j, 0.50
                corr = 1 - (dist / 2)
                return i, j, corr

            results = Parallel(n_jobs=self.config['performance']['n_jobs'])(
                delayed(calc_pair)(i, j) for i, j in pairs
            )

            for i, j, corr in results:
                corr_matrix[i, j] = corr
                corr_matrix[j, i] = corr

        elif method == 'sector_proxy_market':
            for i in range(n):
                for j in range(i+1, n):
                    market_corr_i = self.calculate_market_correlation(
                        returns_df, tickers[i], market_returns, reference_date
                    )
                    market_corr_j = self.calculate_market_correlation(
                        returns_df, tickers[j], market_returns, reference_date
                    )
                    avg_market_corr = np.nanmean([market_corr_i, market_corr_j])

                    corr = self.get_sector_proxy_correlation(
                        tickers[i], tickers[j], sector_map, avg_market_corr
                    )
                    corr_matrix[i, j] = corr
                    corr_matrix[j, i] = corr
        else:
            for i in range(n):
                for j in range(i+1, n):
                    if method == 'ewma':
                        corr = self.calculate_ewma_correlation(
                            returns_df, tickers[i], tickers[j], reference_date
                        )
                    elif method == 'sma':
                        corr = self.calculate_sma_correlation(
                            returns_df, tickers[i], tickers[j], reference_date
                        )
                    else:
                        corr = 0.50

                    if np.isnan(corr):
                        corr = 0.50

                    corr_matrix[i, j] = corr
                    corr_matrix[j, i] = corr

        return corr_matrix

    def build_full_distance_matrix(self, returns_df, all_tickers, reference_date,
                                    sector_map=None, market_returns=None):
        """Построение полной матрицы расстояний"""
        n = len(all_tickers)
        distance_matrix = np.zeros((n, n))

        for i in range(n):
            for j in range(i+1, n):
                ticker_i = all_tickers[i]
                ticker_j = all_tickers[j]

                try:
                    corr = self.calculate_ewma_correlation(
                        returns_df, ticker_i, ticker_j, reference_date
                    )
                except Exception as e:
                    logger.warning(f"Ошибка корреляции {ticker_i}-{ticker_j}: {e}")
                    corr = 0.50

                if np.isnan(corr):
                    corr = self.get_sector_proxy_correlation(
                        ticker_i, ticker_j, sector_map
                    )

                distance = np.sqrt(2 * (1 - corr))
                distance_matrix[i, j] = distance
                distance_matrix[j, i] = distance

        return distance_matrix

    def calculate_average_correlation(self, correlation_matrices):
        """Корректный расчёт средней корреляции"""
        if not correlation_matrices:
            return 0.50

        total_weight = 0
        weighted_sum = 0

        for group, data in correlation_matrices.items():
            matrix = data['matrix']
            n = matrix.shape[0]

            weight = n * (n - 1) / 2
            upper_tri = matrix[np.triu_indices(n, k=1)]
            avg_corr = np.nanmean(upper_tri)

            weighted_sum += avg_corr * weight
            total_weight += weight

        if total_weight > 0:
            return weighted_sum / total_weight
        else:
            return 0.50

## 6. КЛАСТЕРИЗАЦИЯ

In [7]:
class ClusterAnalyzer:
    """Анализ и выбор оптимального числа кластеров"""

    def __init__(self, config):
        self.config = config
        logger.info("ClusterAnalyzer инициализирован")

    def find_optimal_k(self, distance_matrix, tickers, k_range=None):
        """Поиск оптимального k по метрикам"""
        n_samples = len(tickers)

        if k_range is None:
            k_range = range(self.config['clustering']['k_min'],
                          self.config['clustering']['k_max'] + 1)

        k_range = [k for k in k_range if 2 <= k <= n_samples - 1]

        if not k_range:
            k_range = [2] if n_samples > 2 else [1]

        results = []

        for k in k_range:
            try:
                clustering = AgglomerativeClustering(
                    n_clusters=k,
                    metric='precomputed',
                    linkage='average'
                )
                labels = clustering.fit_predict(distance_matrix)

                silhouette = silhouette_score(distance_matrix, labels, metric='precomputed')
                db_index = davies_bouldin_score(distance_matrix, labels)

                results.append({
                    'k': k,
                    'silhouette': silhouette,
                    'davies_bouldin': db_index,
                    'labels': labels
                })
            except Exception as e:
                logger.warning(f"Ошибка кластеризации для k={k}: {e}")
                continue

        if not results:
            logger.error("Не удалось выполнить кластеризацию ни для одного k")
            return None, []

        valid_results = [r for r in results
                        if r['silhouette'] > self.config['clustering']['silhouette_threshold']]

        if valid_results:
            optimal = min(valid_results, key=lambda x: x['davies_bouldin'])
        else:
            optimal = min(results, key=lambda x: abs(x['k'] - self.config['clustering']['k_target']))

        logger.info(f"Оптимальное k={optimal['k']}, Silhouette={optimal['silhouette']:.3f}")

        return optimal, results

    def map_clusters_to_reference(self, labels, reference_labels,
                                   distance_matrix, tickers, reference_tickers,
                                   full_distance_matrix=None, all_tickers=None):
        """Маппинг кластеров на эталонные"""
        unique_labels = np.unique(labels)
        unique_ref = np.unique(reference_labels)

        mapping = {}

        if full_distance_matrix is not None and all_tickers is not None:
            for label in unique_labels:
                mask = labels == label
                cluster_tickers = [tickers[i] for i in range(len(tickers)) if mask[i]]

                best_ref = None
                best_distance = np.inf

                for ref_label in unique_ref:
                    ref_ticker_indices = [i for i, t in enumerate(reference_tickers)
                                         if reference_labels[i] == ref_label]
                    ref_tickers_in_cluster = [reference_tickers[i] for i in ref_ticker_indices]

                    distances = []
                    for t1 in cluster_tickers:
                        for t2 in ref_tickers_in_cluster:
                            try:
                                idx1 = all_tickers.index(t1)
                                idx2 = all_tickers.index(t2)
                                distances.append(full_distance_matrix[idx1, idx2])
                            except ValueError:
                                continue

                    if distances:
                        avg_distance = np.mean(distances)
                        if avg_distance < best_distance:
                            best_distance = avg_distance
                            best_ref = ref_label

                if best_ref is not None:
                    mapping[label] = best_ref
                else:
                    mapping[label] = unique_ref[0]
        else:
            for label in unique_labels:
                mask = labels == label
                cluster_indices = [i for i in range(len(tickers)) if mask[i]]

                best_ref = None
                best_distance = np.inf

                for ref_label in unique_ref:
                    ref_indices = [i for i in range(len(reference_tickers))
                                  if reference_labels[i] == ref_label]

                    if not ref_indices:
                        continue

                    if len(tickers) == len(reference_tickers):
                        distances = []
                        for i in cluster_indices:
                            for j in ref_indices:
                                if i < len(distance_matrix) and j < len(distance_matrix[i]):
                                    distances.append(distance_matrix[i, j])

                        if distances:
                            avg_distance = np.mean(distances)
                            if avg_distance < best_distance:
                                best_distance = avg_distance
                                best_ref = ref_label

                if best_ref is not None:
                    mapping[label] = best_ref
                else:
                    mapping[label] = unique_ref[0]

        mapped_labels = np.array([mapping[l] for l in labels])

        logger.info(f"Маппинг кластеров: {len(mapping)} кластеров mapped")

        return mapped_labels

    def should_recluster(self, last_cluster_date, current_date, frequency_days):
        """
        Проверка необходимости перекластеризации
        """
        if last_cluster_date is None:
            return True

        days_diff = (current_date - last_cluster_date).days
        return days_diff >= frequency_days

## 7. ОБРАБОТКА НЕЛИКВИДНЫХ АКЦИЙ

In [8]:
class IlliquidHandler:
    """Обработка неликвидных акций"""

    def __init__(self, config):
        self.config = config
        logger.info("IlliquidHandler инициализирован")

    def impute_missing_returns(self, returns_df, market_returns, ticker,
                                reference_date, beta=None):
        """Импутация пропусков через Market Model"""
        data = returns_df[ticker].loc[:reference_date]
        missing_mask = data.isna()

        if not missing_mask.any():
            return data, beta

        if beta is None:
            available_mask = ~missing_mask
            if available_mask.sum() > 50:
                y = data[available_mask].values
                x = market_returns.reindex(data.index[available_mask]).fillna(0).values
                try:
                    beta = np.cov(x, y, ddof=1)[0, 1] / np.var(x, ddof=1)
                    beta = np.clip(beta,
                                  self.config['liquidity']['imputation_beta_clip'][0],
                                  self.config['liquidity']['imputation_beta_clip'][1])
                except:
                    beta = 1.0
                    logger.warning(f"Не удалось оценить Beta для {ticker}, используем 1.0")
            else:
                beta = 1.0
                logger.warning(f"Недостаточно данных для Beta {ticker}, используем 1.0")

        imputed = data.copy()
        market_returns_aligned = market_returns.reindex(data.index).fillna(0)
        imputed[missing_mask] = beta * market_returns_aligned[missing_mask]

        logger.info(f"Импутация для {ticker}: {missing_mask.sum()} пропусков, Beta={beta:.2f}")

        return imputed, beta

    def get_liquidity_multiplier(self, liquidity_ratio):
        """Получение множителя ликвидности"""
        premium = self.config['liquidity']['liquidity_premium']

        for threshold, multiplier in sorted(premium.items(), reverse=True):
            if liquidity_ratio >= threshold:
                return multiplier

        return premium[0.0]

    def apply_liquidity_premium(self, sigma, liquidity_ratio):
        """Применение Liquidity Premium к волатильности"""
        multiplier = self.get_liquidity_multiplier(liquidity_ratio)
        return sigma * multiplier

    def cap_correlation(self, corr_matrix, liquid_indices, illiquid_indices, cap=None):
        """Ограничение liquid-illiquid корреляций"""
        if cap is None:
            cap = self.config['liquidity']['correlation_cap']

        capped_matrix = corr_matrix.copy()

        for i in liquid_indices:
            for j in illiquid_indices:
                if i < corr_matrix.shape[0] and j < corr_matrix.shape[1]:
                    capped_matrix[i, j] = min(corr_matrix[i, j], cap)
                    capped_matrix[j, i] = min(corr_matrix[i, j], cap)

        logger.info(f"Ограничение liquid-illiquid корреляций: cap={cap}")

        return capped_matrix

## 8. АНАЛИЗ УСТОЙЧИВОСТИ

In [9]:
class StabilityAnalyzer:
    """Анализ устойчивости кластеризации во времени"""

    def __init__(self, config):
        self.config = config
        logger.info("StabilityAnalyzer инициализирован")

    def calculate_ari(self, labels1, labels2):
        """Расчёт Adjusted Rand Index"""
        return adjusted_rand_score(labels1, labels2)

    def check_stability(self, cluster_history, window_quarters=8):
        """Проверка устойчивости по кварталам"""
        if len(cluster_history) < 2:
            return {'stable': True, 'ari_mean': 1.0, 'ari_min': 1.0, 'ari_scores': []}

        ari_scores = []
        for i in range(1, len(cluster_history)):
            ari = self.calculate_ari(cluster_history[i-1], cluster_history[i])
            ari_scores.append(ari)

        ari_mean = np.mean(ari_scores)
        ari_min = np.min(ari_scores)

        stable = ari_mean >= self.config['clustering']['ari_threshold']

        logger.info(f"Устойчивость: ARI Mean={ari_mean:.3f}, ARI Min={ari_min:.3f}, Stable={stable}")

        return {
            'stable': stable,
            'ari_mean': ari_mean,
            'ari_min': ari_min,
            'ari_scores': ari_scores
        }

## 9. ОСНОВНОЙ КЛАСС КЛАСТЕРИЗАЦИИ (С 4 РЕЖИМАМИ)

In [10]:
class AssetClustering:
    """
    Основной класс кластеризации активов
    Поддерживает 4 режима для Production, архитектура для 9 режимов
    """

    def __init__(self, config, imoex_data):
        self.config = config
        self.imoex_data = imoex_data
        self.classifier = DataClassifier(config)
        self.regime_detector = RegimeDetector(imoex_data, config)
        self.corr_calculator = CorrelationCalculator(config)
        self.cluster_analyzer = ClusterAnalyzer(config)
        self.illiquid_handler = IlliquidHandler(config)
        self.stability_analyzer = StabilityAnalyzer(config)

        self.cluster_history = []
        self.last_reference_date = None
        self.last_cluster_date = None  # ← Для квартальной перекластеризации
        self.crisis_mode = False
        self.crisis_freeze_until = None
        self.frozen_clusters = None
        self.current_regime = None
        self.current_regime_params = None

        logger.info("AssetClustering инициализирован (4 режима)")
        logger.info(f"   Активные режимы: {config['regimes']['active']}")
        logger.info(f"   Частота перекластеризации: {config['clustering']['recluster_frequency_days']} дней")

    def run(self, returns_df, sector_map, reference_date,
            vix_data=None, use_vix=False, force_recluster=False):
        """
        Полный пайплайн кластеризации с 4 режимами

        Parameters
        ----------
        force_recluster : bool
            Принудительная перекластеризация (игнорирует частоту)
        """
        logger.info(f"\n{'='*80}")
        logger.info(f"КЛАСТЕРИЗАЦИЯ АКТИВОВ на {reference_date}")
        logger.info(f"{'='*80}")

        # 1. Валидация данных
        self._validate_input_data(returns_df, reference_date)

        # 2. Классификация
        history_groups = self._classify_assets(returns_df, reference_date)
        liquid_tickers, illiquid_tickers = self._classify_liquidity(returns_df, reference_date)

        # 3. Определение режима (4 режима)
        regime, regime_metrics = self._detect_regime(reference_date, use_vix, vix_data, returns_df)

        # 4. Импутация
        returns_imputed, beta_map = self._impute_illiquid(
            returns_df, illiquid_tickers, reference_date
        )

        # 5. Корреляционные матрицы
        correlation_matrices = self._calculate_correlations(
            returns_imputed, history_groups, liquid_tickers, illiquid_tickers,
            reference_date, sector_map, regime
        )

        # 6. Проверка кризиса
        avg_corr = self.corr_calculator.calculate_average_correlation(correlation_matrices)
        if self._check_crisis_mode(reference_date, regime_metrics, avg_corr):
            logger.warning("КРИЗИСНЫЙ РЕЖИМ: кластеры зафиксированы")
            return self._get_frozen_clusters(reference_date)

        # 7. ← Проверка необходимости перекластеризации
        should_recluster = force_recluster or self.cluster_analyzer.should_recluster(
            self.last_cluster_date,
            reference_date,
            self.config['clustering']['recluster_frequency_days']
        )

        if should_recluster:
            logger.info("Перекластеризация: выполняется полная кластеризация")
            self.last_cluster_date = reference_date

            # Полная матрица расстояний для маппинга
            all_tickers = list(returns_df.columns)
            full_distance_matrix = self.corr_calculator.build_full_distance_matrix(
                returns_imputed, all_tickers, reference_date, sector_map,
                self.imoex_data['close'].pct_change()
            )

            # Кластеризация
            cluster_results = self._perform_clustering(
                history_groups, correlation_matrices, full_distance_matrix,
                all_tickers, sector_map, reference_date
            )
        else:
            logger.info("Перекластеризация: используется предыдущая кластеризация")
            # Используем предыдущие кластеры, обновляем только параметры
            cluster_results = self._get_previous_clusters(reference_date)

        # 8. Применение параметров режима (ежедневное обновление)
        conservative_params = self._apply_regime_parameters(
            history_groups, returns_imputed, beta_map, illiquid_tickers,
            reference_date, regime
        )

        # 9. Анализ устойчивости
        stability = self._analyze_stability(cluster_results, list(returns_df.columns))

        # Сохранение состояния
        self.last_reference_date = reference_date
        self.current_regime = regime
        self.current_regime_params = self.regime_detector.get_regime_parameters(regime)

        # Формирование результатов
        results = self._compile_results(
            reference_date, regime, regime_metrics, history_groups,
            liquid_tickers, illiquid_tickers, cluster_results,
            correlation_matrices, conservative_params, stability,
            beta_map, avg_corr, should_recluster
        )

        logger.info(f"\n{'='*80}")
        logger.info("КЛАСТЕРИЗАЦИЯ ЗАВЕРШЕНА")
        logger.info(f"{'='*80}")

        return results

    def _validate_input_data(self, returns_df, reference_date):
        """Валидация входных данных"""
        logger.info("[1/9] Валидация данных...")
        validation = self.classifier.validate_data(returns_df, reference_date)
        if not validation['valid']:
            logger.error(f"Валидация не пройдена: {validation['errors']}")
            raise ValueError(f"Валидация данных не пройдена: {validation['errors']}")
        for warning in validation['warnings']:
            logger.warning(f"Валидация: {warning}")

    def _classify_assets(self, returns_df, reference_date):
        """Классификация по истории"""
        logger.info("[2/9] Классификация по истории...")
        history_groups = self.classifier.classify_by_history(returns_df, reference_date)
        for group, tickers in history_groups.items():
            logger.info(f"   Группа {group}: {len(tickers)} акций")
        return history_groups

    def _classify_liquidity(self, returns_df, reference_date):
        """Классификация по ликвидности"""
        logger.info("[3/9] Классификация по ликвидности...")
        liquid_tickers, illiquid_tickers = self.classifier.classify_by_liquidity(
            returns_df, reference_date
        )
        logger.info(f"   Ликвидные: {len(liquid_tickers)}, Неликвидные: {len(illiquid_tickers)}")
        return liquid_tickers, illiquid_tickers

    def _detect_regime(self, reference_date, use_vix, vix_data, returns_df):
        """Определение режима (4 режима)"""
        logger.info("[4/9] Определение режима (4 режима)...")
        market_returns = returns_df.mean(axis=1)  # Упрощённо для демонстрации
        regime, regime_metrics = self.regime_detector.detect_regime(
            reference_date, vix_data, market_returns
        )
        logger.info(f"   Режим: {regime}")
        logger.info(f"   Active: {regime_metrics['is_active']}")
        return regime, regime_metrics

    def _check_crisis_mode(self, reference_date, regime_metrics, avg_corr):
        """Проверка кризисного режима"""
        logger.info("[4.1/9] Проверка на кризисный режим...")
        if self.crisis_mode:
            if self.crisis_freeze_until and reference_date > self.crisis_freeze_until:
                self.crisis_mode = False
                self.crisis_freeze_until = None
                self.frozen_clusters = None
                logger.info("Кризисный режим снят, frozen_clusters очищен")
                return False
            return True

        if regime_metrics and self.regime_detector:
            is_crisis, crisis_signals = self.regime_detector.is_crisis(
                regime_metrics, avg_corr
            )

            if is_crisis:
                self.crisis_mode = True
                self.crisis_freeze_until = reference_date + timedelta(
                    days=self.config['crisis']['freeze_duration_days']
                )
                logger.warning(f"Кризисный режим активирован до {self.crisis_freeze_until}")
                return True

        return False

    def _impute_illiquid(self, returns_df, illiquid_tickers, reference_date):
        """Импутация для неликвидных"""
        logger.info("[5/9] Импутация для неликвидных акций...")
        returns_imputed = returns_df.copy()
        beta_map = {}

        market_returns = self.imoex_data['close'].pct_change()

        for ticker in illiquid_tickers:
            imputed, beta = self.illiquid_handler.impute_missing_returns(
                returns_imputed, market_returns,
                ticker, reference_date
            )
            returns_imputed.loc[:reference_date, ticker] = imputed
            beta_map[ticker] = beta

        logger.info(f"   Импутация выполнена для {len(illiquid_tickers)} акций")

        return returns_imputed, beta_map

    def _calculate_correlations(self, returns_imputed, history_groups,
                                 liquid_tickers, illiquid_tickers,
                                 reference_date, sector_map, regime):
        """Расчёт корреляционных матриц с корректировкой по режиму"""
        logger.info("[6/9] Расчёт корреляционных матриц...")
        correlation_matrices = {}
        market_returns = self.imoex_data['close'].pct_change()

        # Получение параметров режима
        regime_params = self.regime_detector.get_regime_parameters(regime)

        for group, tickers in history_groups.items():
            if len(tickers) < 2:
                logger.warning(f"   Группа {group}: недостаточно акций для корреляции")
                continue

            method = self.config['history_groups'][group]['method']
            corr_matrix = self.corr_calculator.build_correlation_matrix(
                returns_imputed, tickers, reference_date, method, sector_map,
                market_returns if method == 'sector_proxy_market' else None
            )

            # ← Корректировка корреляций по режиму
            corr_matrix = self.corr_calculator.adjust_correlations_for_regime(
                corr_matrix, regime_params
            )

            if group in ['B', 'C', 'D'] and illiquid_tickers:
                liquid_idx = [i for i, t in enumerate(tickers) if t in liquid_tickers]
                illiquid_idx = [i for i, t in enumerate(tickers) if t in illiquid_tickers]
                corr_matrix = self.illiquid_handler.cap_correlation(
                    corr_matrix, liquid_idx, illiquid_idx
                )

            correlation_matrices[group] = {
                'tickers': tickers,
                'matrix': corr_matrix,
                'method': method
            }
            logger.info(f"   Группа {group}: {method}, матрица {len(tickers)}x{len(tickers)}")

        return correlation_matrices

    def _perform_clustering(self, history_groups, correlation_matrices,
                            full_distance_matrix, all_tickers,
                            sector_map, reference_date):
        """Кластеризация"""
        logger.info("[7/9] Кластеризация...")
        cluster_results = {}

        if 'D' in history_groups and len(history_groups['D']) >= 2:
            tickers_d = history_groups['D']
            corr_d = correlation_matrices['D']['matrix']

            distance_d = np.sqrt(2 * (1 - corr_d))
            distance_d = np.clip(distance_d, 0, 2)
            distance_d = np.nan_to_num(distance_d, nan=0.5)

            optimal_k, all_results = self.cluster_analyzer.find_optimal_k(
                distance_d, tickers_d
            )

            if optimal_k:
                cluster_results['D'] = {
                    'labels': optimal_k['labels'],
                    'k': optimal_k['k'],
                    'silhouette': optimal_k['silhouette'],
                    'tickers': tickers_d
                }
                logger.info(f"   Группа D: k={optimal_k['k']}, Silhouette={optimal_k['silhouette']:.3f}")

        for group in ['A', 'B', 'C']:
            if group not in history_groups or len(history_groups[group]) < 2:
                continue

            tickers_g = history_groups[group]

            if 'D' in cluster_results:
                ref_labels = cluster_results['D']['labels']
                ref_tickers = cluster_results['D']['tickers']

                corr_g = correlation_matrices.get(group, {}).get('matrix', np.eye(len(tickers_g)))
                distance_g = np.sqrt(2 * (1 - corr_g))
                distance_g = np.clip(distance_g, 0, 2)
                distance_g = np.nan_to_num(distance_g, nan=0.5)

                optimal_k_g, _ = self.cluster_analyzer.find_optimal_k(
                    distance_g, tickers_g,
                    k_range=range(2, min(8, len(tickers_g)))
                )

                if optimal_k_g:
                    mapped_labels = self.cluster_analyzer.map_clusters_to_reference(
                        optimal_k_g['labels'], ref_labels,
                        distance_g, tickers_g, ref_tickers,
                        full_distance_matrix=full_distance_matrix,
                        all_tickers=all_tickers
                    )

                    cluster_results[group] = {
                        'labels': mapped_labels,
                        'k': len(np.unique(mapped_labels)),
                        'mapped_to_D': True,
                        'tickers': tickers_g
                    }
                else:
                    temp_labels = np.array([sector_map.get(t, 0) for t in tickers_g])
                    cluster_results[group] = {
                        'labels': temp_labels,
                        'k': len(np.unique(temp_labels)),
                        'mapped_to_D': False,
                        'tickers': tickers_g
                    }
            else:
                corr_g = correlation_matrices[group]['matrix']
                distance_g = np.sqrt(2 * (1 - corr_g))
                distance_g = np.nan_to_num(distance_g, nan=0.5)

                optimal_k, _ = self.cluster_analyzer.find_optimal_k(
                    distance_g, tickers_g,
                    k_range=range(2, min(8, len(tickers_g)))
                )

                if optimal_k:
                    cluster_results[group] = {
                        'labels': optimal_k['labels'],
                        'k': optimal_k['k'],
                        'tickers': tickers_g
                    }
                else:
                    temp_labels = np.array([sector_map.get(t, 0) for t in tickers_g])
                    cluster_results[group] = {
                        'labels': temp_labels,
                        'k': len(np.unique(temp_labels)),
                        'tickers': tickers_g
                    }

            logger.info(f"   Группа {group}: k={cluster_results[group]['k']}")

        return cluster_results

    def _get_previous_clusters(self, reference_date):
        """Возврат предыдущих кластеров (без перекластеризации)"""
        if not self.cluster_history:
            logger.warning("Нет предыдущих кластеров, возвращаем пустые")
            return {}

        # Возвращаем последние кластеры
        return self.cluster_history[-1].get('cluster_results', {})

    def _apply_regime_parameters(self, history_groups, returns_imputed,
                                  beta_map, illiquid_tickers, reference_date, regime):
        """
        Применение параметров режима (ежедневное обновление)
        """
        logger.info("[8/9] Применение параметров режима...")
        conservative_params = {}

        regime_params = self.regime_detector.get_regime_parameters(regime)

        for group, tickers in history_groups.items():
            config_key = f'group_{group}'
            if config_key not in self.config['conservative']:
                logger.warning(f"Группа {group} отсутствует в conservative config, используем fallback")
                mult = {'beta_multiplier': 1.0, 'sigma_multiplier': 1.0}
            else:
                mult = self.config['conservative'][config_key]

            for ticker in tickers:
                base_beta = beta_map.get(ticker, 1.0) if ticker in illiquid_tickers else 1.0
                base_sigma = returns_imputed[ticker].loc[:reference_date].std()

                liq_ratio = self.classifier.get_liquidity_ratio(
                    returns_imputed, ticker, reference_date
                )
                adjusted_sigma = self.illiquid_handler.apply_liquidity_premium(
                    base_sigma, liq_ratio
                )

                # ← Применение множителей режима
                regime_sigma_mult = regime_params.get('sigma_multiplier', 1.0)
                regime_haircut_mult = regime_params.get('haircut_multiplier', 1.0)

                conservative_params[ticker] = {
                    'beta_base': base_beta,
                    'beta_adjusted': base_beta * mult['beta_multiplier'],
                    'sigma_base': base_sigma,
                    'sigma_adjusted': adjusted_sigma * mult['sigma_multiplier'] * regime_sigma_mult,
                    'beta_multiplier': mult['beta_multiplier'],
                    'sigma_multiplier': mult['sigma_multiplier'] * regime_sigma_mult,
                    'haircut_multiplier': regime_haircut_mult,
                    'history_group': group,
                    'liquidity_ratio': liq_ratio,
                    'regime': regime
                }

        return conservative_params

    def _analyze_stability(self, cluster_results, all_tickers):
        """Анализ устойчивости"""
        logger.info("[8.1/9] Анализ устойчивости...")
        current_labels = self._merge_cluster_results(cluster_results, all_tickers)

        # Сохраняем полную информацию для истории
        cluster_entry = {
            'cluster_results': cluster_results,
            'labels': current_labels,
            'date': self.last_reference_date
        }
        self.cluster_history.append(cluster_entry)

        stability = self.stability_analyzer.check_stability(self.cluster_history)
        logger.info(f"   ARI Mean: {stability['ari_mean']:.3f}")
        logger.info(f"   ARI Min: {stability['ari_min']:.3f}")
        logger.info(f"   Stable: {stability['stable']}")

        return stability

    def _compile_results(self, reference_date, regime, regime_metrics,
                         history_groups, liquid_tickers, illiquid_tickers,
                         cluster_results, correlation_matrices,
                         conservative_params, stability, beta_map, avg_corr,
                         should_recluster):
        """Формирование результатов"""
        logger.info("[9/9] Формирование результатов...")

        results = {
            'reference_date': reference_date,
            'regime': regime,
            'regime_metrics': regime_metrics,
            'history_groups': history_groups,
            'liquid_tickers': liquid_tickers,
            'illiquid_tickers': illiquid_tickers,
            'cluster_results': cluster_results,
            'correlation_matrices': correlation_matrices,
            'conservative_params': conservative_params,
            'stability': stability,
            'beta_map': beta_map,
            'crisis_mode': self.crisis_mode,
            'avg_correlation': avg_corr,
            'should_recluster': should_recluster,
            'last_cluster_date': self.last_cluster_date
        }

        return results

    def _merge_cluster_results(self, cluster_results, all_tickers):
        """Объединение результатов кластеризации по группам"""
        labels = np.zeros(len(all_tickers), dtype=int)

        for group, result in cluster_results.items():
            for i, ticker in enumerate(result['tickers']):
                if ticker in all_tickers:
                    try:
                        idx = list(all_tickers).index(ticker)
                        labels[idx] = result['labels'][i]
                    except ValueError:
                        logger.warning(f"Тикер {ticker} не найден в all_tickers")

        return labels

    def _get_frozen_clusters(self, reference_date):
        """Возврат зафиксированных кластеров в кризис"""
        if not self.cluster_history:
            logger.warning("Нет истории кластеров для заморозки")
            return None

        return {
            'reference_date': reference_date,
            'frozen': True,
            'labels': self.cluster_history[-1]['labels'],
            'crisis_mode': True
        }

## 10. ЭКСПОРТ РЕЗУЛЬТАТОВ

In [11]:
class ResultsExporter:
    """Экспорт результатов кластеризации"""

    def __init__(self, output_dir='./clustering_output'):
        self.output_dir = output_dir
        import os
        os.makedirs(output_dir, exist_ok=True)
        logger.info(f"ResultsExporter инициализирован: {output_dir}")

    def export_clusters(self, results, all_tickers):
        """Экспорт карты кластеров"""
        df = pd.DataFrame({
            'ticker': all_tickers,
            'cluster_id': results.get('cluster_results', {}).get('D', {}).get('labels', np.zeros(len(all_tickers))),
            'history_group': [results['conservative_params'].get(t, {}).get('history_group', 'Unknown')
                             for t in all_tickers],
            'liquidity_class': ['Illiquid' if t in results.get('illiquid_tickers', []) else 'Liquid'
                               for t in all_tickers],
            'beta_base': [results['conservative_params'].get(t, {}).get('beta_base', 1.0)
                         for t in all_tickers],
            'beta_adjusted': [results['conservative_params'].get(t, {}).get('beta_adjusted', 1.0)
                             for t in all_tickers],
            'sigma_base': [results['conservative_params'].get(t, {}).get('sigma_base', 0.0)
                          for t in all_tickers],
            'sigma_adjusted': [results['conservative_params'].get(t, {}).get('sigma_adjusted', 0.0)
                              for t in all_tickers],
            'liquidity_ratio': [results['conservative_params'].get(t, {}).get('liquidity_ratio', 1.0)
                               for t in all_tickers],
            'regime': [results['conservative_params'].get(t, {}).get('regime', 'Unknown')
                      for t in all_tickers],
            'haircut_multiplier': [results['conservative_params'].get(t, {}).get('haircut_multiplier', 1.0)
                                  for t in all_tickers],
            'last_update_date': results['reference_date']
        })

        filepath = f"{self.output_dir}/cluster_mapping.csv"
        df.to_csv(filepath, index=False)
        logger.info(f"Экспорт: {filepath}")

        return df

    def export_metrics(self, results):
        """Экспорт метрик качества"""
        metrics = []

        for group, cluster_result in results.get('cluster_results', {}).items():
            metrics.append({
                'history_group': group,
                'n_clusters': cluster_result.get('k', 0),
                'silhouette_score': cluster_result.get('silhouette', 0),
                'n_tickers': len(cluster_result.get('tickers', [])),
                'method': results.get('correlation_matrices', {}).get(group, {}).get('method', 'Unknown'),
                'mapped_to_D': cluster_result.get('mapped_to_D', False)
            })

        metrics.append({
            'history_group': 'Overall',
            'n_clusters': 0,
            'silhouette_score': results.get('stability', {}).get('ari_mean', 0),
            'n_tickers': 0,
            'method': f"ARI_Stability_{results.get('stability', {}).get('stable', False)}",
            'mapped_to_D': False
        })

        metrics.append({
            'history_group': 'Regime',
            'n_clusters': 0,
            'silhouette_score': 0,
            'n_tickers': 0,
            'method': results.get('regime', 'Unknown'),
            'mapped_to_D': False
        })

        df = pd.DataFrame(metrics)
        filepath = f"{self.output_dir}/clustering_metrics.csv"
        df.to_csv(filepath, index=False)
        logger.info(f"Экспорт: {filepath}")

        return df

    def export_config(self, config):
        """Экспорт конфигурации"""
        filepath = f"{self.output_dir}/clustering_config.yaml"
        with open(filepath, 'w', encoding='utf-8') as f:
            yaml.dump(config, f, allow_unicode=True, default_flow_style=False)
        logger.info(f"Экспорт: {filepath}")

    def export_visualization(self, results, all_tickers, corr_matrix=None):
        """Экспорт визуализации"""
        plt.style.use('seaborn-v0_8-whitegrid')
        fig = plt.figure(figsize=(24, 14))

        # 1. Распределение по группам истории
        ax1 = plt.subplot(3, 3, 1)
        group_sizes = [len(results['history_groups'][g]) for g in ['A', 'B', 'C', 'D']]
        ax1.bar(['A (<30)', 'B (30-100)', 'C (100-200)', 'D (200+)'], group_sizes,
               color=['red', 'orange', 'yellow', 'green'], alpha=0.7)
        ax1.set_title('Распределение по группам истории', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Количество акций')

        # 2. Ликвидность
        ax2 = plt.subplot(3, 3, 2)
        n_liquid = len(results.get('liquid_tickers', []))
        n_illiquid = len(results.get('illiquid_tickers', []))
        ax2.bar(['Ликвидные', 'Неликвидные'], [n_liquid, n_illiquid],
               color=['green', 'red'], alpha=0.7)
        ax2.set_title('Распределение по ликвидности', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Количество акций')

        # 3. Режим (4 режима)
        ax3 = plt.subplot(3, 3, 3)
        regime = results.get('regime', 'UNKNOWN')
        regime_colors = {
            'BULL_LOW': 'green', 'BULL_HIGH': 'lightgreen',
            'BEAR_LOW': 'red', 'BEAR_HIGH': 'lightcoral'
        }
        ax3.barh(['Режим'], [1], color=regime_colors.get(regime, 'gray'))
        ax3.set_title(f'Рыночный режим: {regime}', fontsize=12, fontweight='bold')
        ax3.set_xlim(0, 1)

        # 4. Устойчивость ARI
        ax4 = plt.subplot(3, 3, 4)
        ari_scores = results.get('stability', {}).get('ari_scores', [])
        if ari_scores:
            ax4.plot(range(len(ari_scores)), ari_scores, marker='o', linewidth=2)
            ax4.axhline(y=0.7, color='red', linestyle='--', label='Порог 0.7')
        ax4.set_title('Динамика ARI по кварталам', fontsize=12, fontweight='bold')
        ax4.set_ylabel('ARI')
        ax4.set_xlabel('Квартал')
        ax4.legend()

        # 5. Параметры режима
        ax5 = plt.subplot(3, 3, 5)
        regime_params = results['conservative_params'][all_tickers[0]] if all_tickers else {}
        param_names = ['sigma_multiplier', 'haircut_multiplier']
        param_values = [regime_params.get(p, 1.0) for p in param_names]
        ax5.bar(param_names, param_values, alpha=0.7, color='steelblue')
        ax5.set_title('Параметры режима', fontsize=12, fontweight='bold')
        ax5.set_ylabel('Multiplier')

        # 6. Консервативные параметры (Sigma)
        ax6 = plt.subplot(3, 3, 6)
        sigma_mults = [results['conservative_params'][t]['sigma_multiplier']
                      for t in all_tickers[:20]]
        ax6.bar(range(len(sigma_mults)), sigma_mults, alpha=0.7, color='coral')
        ax6.set_title('Sigma множители (первые 20 акций)', fontsize=12, fontweight='bold')
        ax6.set_ylabel('Multiplier')

        # 7. Scores режимов
        ax7 = plt.subplot(3, 3, 7)
        regime_metrics = results.get('regime_metrics', {})
        scores = [regime_metrics.get('direction_score', 0), regime_metrics.get('volatility_score', 0)]
        ax7.bar(['Direction', 'Volatility'], scores, alpha=0.7, color=['blue', 'orange'])
        ax7.axhline(y=0.6, color='green', linestyle='--', label='Bull порог')
        ax7.axhline(y=0.4, color='red', linestyle='--', label='Bear порог')
        ax7.set_title('Scores режимов', fontsize=12, fontweight='bold')
        ax7.set_ylabel('Score')
        ax7.legend()

        # 8. Heatmap корреляционной матрицы
        ax8 = plt.subplot(3, 3, 8)
        if corr_matrix is not None:
            n = min(20, corr_matrix.shape[0])
            sns.heatmap(corr_matrix[:n, :n], cmap='RdYlGn', vmin=-1, vmax=1,
                       ax=ax8, annot=False, square=True)
            ax8.set_title(f'Heatmap корреляций (первые {n} акций)', fontsize=12, fontweight='bold')
        else:
            ax8.text(0.5, 0.5, 'Нет данных', ha='center', va='center')

        # 9. Дендрограмма
        ax9 = plt.subplot(3, 3, 9)
        if corr_matrix is not None:
            n = min(20, corr_matrix.shape[0])
            distance_matrix = np.sqrt(2 * (1 - corr_matrix[:n, :n]))
            distance_matrix = np.nan_to_num(distance_matrix, nan=0.5)
            linkage_matrix = linkage(distance_matrix, method='average')
            dendrogram(linkage_matrix, ax=ax9, leaf_rotation=90, leaf_font_size=8)
            ax9.set_title('Дендрограмма кластеров', fontsize=12, fontweight='bold')
            ax9.set_ylabel('Расстояние')
        else:
            ax9.text(0.5, 0.5, 'Нет данных', ha='center', va='center')

        plt.tight_layout()
        filepath = f"{self.output_dir}/clustering_visualization.png"
        plt.savefig(filepath, dpi=150, bbox_inches='tight')
        logger.info(f"Экспорт: {filepath}")
        plt.close()

## 11. UNIT-ТЕСТЫ (16 ТЕСТОВ)

In [12]:
logger.info("Запуск unit-тестов...")
print("\n" + "="*80)
print("UNIT-ТЕСТЫ")
print("="*80)

def run_unit_tests():
    """Запуск расширенных unit-тестов"""
    tests_passed = 0
    tests_total = 0

    config = CLUSTERING_CONFIG

    np.random.seed(42)
    n_days = 300
    n_tickers = 20
    dates = pd.date_range('2023-01-01', periods=n_days, freq='B')
    returns = pd.DataFrame(
        np.random.randn(n_days, n_tickers) * 0.02,
        columns=[f'TEST_{i:02d}' for i in range(n_tickers)],
        index=dates
    )

    imoex_data = pd.DataFrame({
        'close': 100 * np.exp(np.cumsum(np.random.randn(n_days) * 0.015))
    }, index=dates)

    # Тест 1: Корреляционная матрица
    print("\n[Тест 1/16] Корреляционная матрица...")
    tests_total += 1
    try:
        calc = CorrelationCalculator(config)
        corr = calc.build_correlation_matrix(returns, list(returns.columns),
                                            returns.index[-1], 'ewma')
        assert np.allclose(corr, corr.T, equal_nan=True), "Матрица не симметрична"
        assert np.allclose(np.diag(corr), 1.0, equal_nan=True), "Диагональ не равна 1"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 2: Классификация по истории
    print("[Тест 2/16] Классификация по истории...")
    tests_total += 1
    try:
        classifier = DataClassifier(config)
        groups = classifier.classify_by_history(returns, returns.index[-1])
        total = sum(len(g) for g in groups.values())
        assert total == n_tickers, "Не все тикеры классифицированы"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 3: Liquidity multiplier
    print("[Тест 3/16] Liquidity multiplier...")
    tests_total += 1
    try:
        handler = IlliquidHandler(config)
        assert handler.get_liquidity_multiplier(0.90) == 1.0
        assert handler.get_liquidity_multiplier(0.50) == 1.5
        assert handler.get_liquidity_multiplier(0.30) == 2.0
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 4: ARI расчёт
    print("[Тест 4/16] ARI расчёт...")
    tests_total += 1
    try:
        analyzer = StabilityAnalyzer(config)
        labels1 = np.array([0, 0, 1, 1, 2])
        labels2 = np.array([0, 0, 1, 1, 2])
        ari = analyzer.calculate_ari(labels1, labels2)
        assert ari == 1.0, f"ARI должен быть 1.0, получен {ari}"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 5: Воспроизводимость
    print("[Тест 5/16] Воспроизводимость...")
    tests_total += 1
    try:
        np.random.seed(42)
        corr1 = calc.build_correlation_matrix(returns, list(returns.columns),
                                             returns.index[-1], 'ewma')
        np.random.seed(42)
        corr2 = calc.build_correlation_matrix(returns, list(returns.columns),
                                             returns.index[-1], 'ewma')
        assert np.allclose(corr1, corr2, equal_nan=True), "Результаты не воспроизводятся"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 6: Wasserstein расстояние
    print("[Тест 6/16] Wasserstein расстояние...")
    tests_total += 1
    try:
        np.random.seed(42)
        dist = calc.calculate_wasserstein_distance(returns, 'TEST_00', 'TEST_01', returns.index[-1])
        assert not np.isnan(dist), "Wasserstein расстояние NaN"
        assert dist >= 0, "Wasserstein расстояние отрицательное"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 7: Импутация пропусков
    print("[Тест 7/16] Импутация пропусков...")
    tests_total += 1
    try:
        returns_with_gaps = returns.copy()
        returns_with_gaps.iloc[10:20, 0] = np.nan

        market_returns = returns['TEST_01'].pct_change().reindex(returns.index).fillna(0)

        imputed, beta = handler.impute_missing_returns(
            returns_with_gaps, market_returns, 'TEST_00', returns.index[-1]
        )
        assert not imputed.isna().any(), "Пропуски не заполнены"
        assert 0.5 <= beta <= 3.0, f"Beta вне диапазона: {beta}"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 8: Маппинг кластеров
    print("[Тест 8/16] Маппинг кластеров...")
    tests_total += 1
    try:
        cluster_analyzer = ClusterAnalyzer(config)
        labels = np.array([0, 0, 1, 1, 2])
        ref_labels = np.array([0, 0, 1, 1, 2])
        distance_matrix = np.random.rand(5, 5)
        distance_matrix = (distance_matrix + distance_matrix.T) / 2
        np.fill_diagonal(distance_matrix, 0)
        tickers = [f'TEST_{i:02d}' for i in range(5)]
        reference_tickers = [f'TEST_{i:02d}' for i in range(5)]
        mapped = cluster_analyzer.map_clusters_to_reference(
            labels, ref_labels, distance_matrix,
            tickers, reference_tickers
        )
        assert len(mapped) == len(labels), "Длина mapped не совпадает"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 9: Валидация данных
    print("[Тест 9/16] Валидация данных...")
    tests_total += 1
    try:
        validation = classifier.validate_data(returns, returns.index[-1])
        assert 'valid' in validation, "Отсутствует поле 'valid'"
        assert 'errors' in validation, "Отсутствует поле 'errors'"
        assert 'warnings' in validation, "Отсутствует поле 'warnings'"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 10: Применение Liquidity Premium
    print("[Тест 10/16] Применение Liquidity Premium...")
    tests_total += 1
    try:
        sigma = 0.02
        adjusted = handler.apply_liquidity_premium(sigma, 0.50)
        assert adjusted > sigma, "Liquidity Premium не увеличивает волатильность"
        assert adjusted == sigma * 1.5, f"Неверный множитель: {adjusted/sigma}"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 11: Определение режима (4 режима)
    print("[Тест 11/16] Определение режима (4 режима)...")
    tests_total += 1
    try:
        regime_detector = RegimeDetector(imoex_data, config)
        regime, metrics = regime_detector.detect_regime(returns.index[-1])
        assert regime in config['regimes']['active'], f"Режим {regime} не активен"
        assert 'direction' in metrics, "Отсутствует direction"
        assert 'volatility' in metrics, "Отсутствует volatility"
        tests_passed += 1
        print(f"PASSED (режим: {regime})")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 12: Проверка на кризис
    print("[Тест 12/16] Проверка на кризис...")
    tests_total += 1
    try:
        metrics = {'volatility_details': {'vix_score': 0.9}, 'imoex_drawdown': -0.25}
        is_crisis, signals = regime_detector.is_crisis(metrics, 0.85)
        assert is_crisis, "Кризис не обнаружен"
        assert len(signals) > 0, "Нет сигналов кризиса"
        tests_passed += 1
        print("PASSED (кризис корректно обнаружен)")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 13: Поиск оптимального k
    print("[Тест 13/16] Поиск оптимального k...")
    tests_total += 1
    try:
        distance_matrix = np.random.rand(10, 10)
        distance_matrix = (distance_matrix + distance_matrix.T) / 2
        np.fill_diagonal(distance_matrix, 0)
        tickers = [f'TEST_{i:02d}' for i in range(10)]
        optimal, all_results = cluster_analyzer.find_optimal_k(
            distance_matrix, tickers
        )
        assert optimal is not None, "Оптимальный k не найден"
        assert 'k' in optimal, "Отсутствует поле 'k'"
        assert 'silhouette' in optimal, "Отсутствует поле 'silhouette'"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 14: Обработка NaN в корреляции
    print("[Тест 14/16] Обработка NaN в корреляции...")
    tests_total += 1
    try:
        returns_with_nan = returns.copy()
        returns_with_nan.iloc[:50, 0] = np.nan
        corr = calc.build_correlation_matrix(returns_with_nan, list(returns_with_nan.columns),
                                            returns_with_nan.index[-1], 'ewma')
        assert not np.all(np.isnan(corr)), "Все корреляции NaN"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 15: Экспорт результатов
    print("[Тест 15/16] Экспорт результатов...")
    tests_total += 1
    try:
        exporter = ResultsExporter('./test_export')
        test_results = {
            'reference_date': datetime.now(),
            'regime': 'BULL_LOW',
            'history_groups': {'A': [], 'B': [], 'C': [], 'D': list(returns.columns)},
            'liquid_tickers': list(returns.columns),
            'illiquid_tickers': [],
            'cluster_results': {'D': {'labels': np.zeros(len(returns.columns)), 'k': 3}},
            'correlation_matrices': {},
            'conservative_params': {t: {'beta_base': 1.0, 'beta_adjusted': 1.0,
                                        'sigma_base': 0.02, 'sigma_adjusted': 0.02,
                                        'beta_multiplier': 1.0, 'sigma_multiplier': 1.0,
                                        'history_group': 'D', 'liquidity_ratio': 1.0,
                                        'regime': 'BULL_LOW', 'haircut_multiplier': 1.0}
                                 for t in returns.columns},
            'stability': {'ari_mean': 0.8, 'stable': True},
            'regime_metrics': {'direction_score': 0.7, 'volatility_score': 0.3}
        }
        df = exporter.export_clusters(test_results, list(returns.columns))
        assert len(df) == len(returns.columns), "Неверное количество строк"
        assert 'cluster_id' in df.columns, "Отсутствует cluster_id"
        assert 'regime' in df.columns, "Отсутствует regime"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Тест 16: Интеграционный тест полного цикла
    print("[Тест 16/16] Интеграционный тест полного цикла...")
    tests_total += 1

    try:
        def example_usage():
            np.random.seed(42)
            n_days = 300
            n_tickers = 20

            tickers = [f'TICKER_{i:03d}' for i in range(n_tickers)]
            dates = pd.date_range('2023-01-01', periods=n_days, freq='B')

            returns_data = np.random.randn(n_days, n_tickers) * 0.02
            returns_df = pd.DataFrame(returns_data, index=dates, columns=tickers)

            for i in range(15, 20):
                n_missing = int(n_days * 0.4)
                missing_idx = np.random.choice(n_days, n_missing, replace=False)
                returns_df.iloc[missing_idx, i] = np.nan

            imoex_data = pd.DataFrame({
                'close': 100 * np.exp(np.cumsum(np.random.randn(n_days) * 0.015))
            }, index=dates)

            sector_map = {}
            sectors = ['Banks', 'OilGas', 'Tech', 'Retail', 'Utilities']
            for i, ticker in enumerate(tickers):
                sector_map[ticker] = sectors[i % len(sectors)]

            clustering = AssetClustering(CLUSTERING_CONFIG, imoex_data)

            results = clustering.run(
                returns_df=returns_df,
                sector_map=sector_map,
                reference_date=dates[-1],
                vix_data=None,
                use_vix=False
            )

            return results

        results = example_usage()
        assert results is not None, "Результаты пустые"
        assert 'regime' in results, "Отсутствует regime"
        assert results['regime'] in CLUSTERING_CONFIG['regimes']['active'], f"Режим {results['regime']} не активен"
        assert len(results['cluster_results']) > 0, "Нет кластеров"
        tests_passed += 1
        print("PASSED")
    except Exception as e:
        print(f"FAILED: {e}")

    # Итоги
    print("\n" + "="*80)
    print(f"РЕЗУЛЬТАТЫ: {tests_passed}/{tests_total} тестов пройдено")
    print("="*80)

    if tests_passed == tests_total:
        print("ВСЕ ТЕСТЫ ПРОЙДЕНЫ")
    else:
        print(f"НЕ ПРОЙДЕНО: {tests_total - tests_passed} тестов")

    return tests_passed == tests_total

# Запуск тестов
tests_passed = run_unit_tests()


UNIT-ТЕСТЫ

[Тест 1/16] Корреляционная матрица...
PASSED
[Тест 2/16] Классификация по истории...
PASSED
[Тест 3/16] Liquidity multiplier...
PASSED
[Тест 4/16] ARI расчёт...
PASSED
[Тест 5/16] Воспроизводимость...


PASSED
[Тест 6/16] Wasserstein расстояние...
PASSED
[Тест 7/16] Импутация пропусков...
PASSED
[Тест 8/16] Маппинг кластеров...
PASSED
[Тест 9/16] Валидация данных...
PASSED
[Тест 10/16] Применение Liquidity Premium...
PASSED
[Тест 11/16] Определение режима (4 режима)...
PASSED (режим: BEAR_HIGH)
[Тест 12/16] Проверка на кризис...
PASSED (кризис корректно обнаружен)
[Тест 13/16] Поиск оптимального k...
PASSED
[Тест 14/16] Обработка NaN в корреляции...


PASSED
[Тест 15/16] Экспорт результатов...
PASSED
[Тест 16/16] Интеграционный тест полного цикла...
PASSED

РЕЗУЛЬТАТЫ: 16/16 тестов пройдено
ВСЕ ТЕСТЫ ПРОЙДЕНЫ


## Перед Production:

**1. Проверить данные по группам:**
```python
# В Production убеждаемся, что есть акции во всех группах A, B, C, D
print(f"Группа A: {len(history_groups['A'])} акций")
print(f"Группа B: {len(history_groups['B'])} акций")
```
**2. Настрить пороги кризиса:**
```yaml
crisis:
  vix_threshold: 40       # VIX-RUS порог
  imoex_drawdown: -0.20   # Просадка IMOEX
  correlation_spike: 0.90 # Средняя корреляция
```

**3. Задокументировать режимы для регулятора:**
- 4 активных режима (Bull/Bear × Low/High)
- Кризисный режим (Freeze на 63 дня)
- Квартальная перекластеризация

**4. Добавить мониторинг:**
- Алерты при переходе в BEAR_HIGH режим
- Трекинг времени в кризисе
- Логирование переключений режимов